# 타이타닉 최종 예측 모델 (인사이트 기반)

검증 노트북에서 얻은 인사이트를 실제 모델에 반영합니다.

앞선 분석의 문제:
- 정확도는 높아 보이지만 재현율이 0.48로 **생존자의 절반을 놓침**
- Age 상관이 낮다고 무관하다고 판단했지만 실제로는 **비선형**
- `Fare==0` 은 무임승차가 아니라 **결측 대체값**
- 모델 순위는 신뢰구간이 겹쳐 **판단 불가**

이번 접근:
1. 인사이트 기반 **피처 엔지니어링** (IsChild, FamilySize, IsAlone, FareIsZero)
2. `Fare==0` 을 결측으로 처리 후 중앙값 대체
3. **클래스 불균형 보정**(class_weight='balanced')
4. **교차검증 + 임계값 조정**으로 재현율 확보

## 0. 준비

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="AppleGothic")

RANDOM_STATE = 42
DATA_PATH = "/Users/remchoi/.cache/kagglehub/datasets/heptapod/titanic/versions/1/train_and_test2.csv"

## 1. 인사이트 기반 피처 엔지니어링

| 피처 | 만든 이유 (인사이트) |
|---|---|
| `IsChild` (Age ≤ 12) | 상관은 -0.06이지만 0~5세 생존율 0.55 → **비선형** |
| `FamilySize` | 4명 부근 생존율 최고, 비선형 |
| `IsAlone` | 1명 생존율 0.21로 낮음 |
| `FareIsZero` | `Fare==0` 17명은 결측 대체값 → **플래그로 분리** |
| `FareLog` | Fare가 한쪽으로 치우친 분포 → 로그 변환 |
| `Fare` (결측 처리) | 0을 NaN으로 바꿔 중앙값으로 대체 |

In [ ]:
raw = pd.read_csv(DATA_PATH).rename(columns={"2urvived": "Survived"})
df = raw.drop(columns=[c for c in raw.columns if c.startswith("zero")] + ["Passengerid"]).copy()
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])


def engineer_features(data):
    x = data.copy()
    x["FamilySize"] = x["sibsp"] + x["Parch"] + 1
    x["IsAlone"] = (x["FamilySize"] == 1).astype(int)
    x["IsChild"] = (x["Age"] <= 12).astype(int)
    x["FareIsZero"] = (x["Fare"] == 0).astype(int)
    x["FareLog"] = np.log1p(x["Fare"])
    x["Fare"] = x["Fare"].replace(0, np.nan)
    return x.drop(columns=["sibsp", "Parch"])


data = engineer_features(df)
X = data.drop(columns="Survived")
y = data["Survived"]

print("피처:", X.columns.tolist())
print("결측치(Fare, 의도적):", X["Fare"].isnull().sum())
print("양성 비율:", round(y.mean(), 3), "| 기준선(전부 사망) 정확도:", round(1 - y.mean(), 3))

## 2. 모델 정의

- **LogReg**: `Fare` 중앙값 대체 + 스케일링 + 클래스 가중치
- **RandomForest**: 중앙값 대체 + 클래스 가중치
- **HistGB**: 결측을 직접 처리(대체 불필요)

In [ ]:
models = {
    "LogReg": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced"),
    ),
    "RandomForest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, max_depth=5,
                               class_weight="balanced", random_state=RANDOM_STATE),
    ),
    "HistGB": HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    ),
}

## 3. 교차검증으로 비교

정확도뿐 아니라 **재현율(recall)** 과 **F1** 을 함께 봅니다.
생존자 탐지가 목적이라면 정확도보다 재현율·F1이 중요합니다.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
metrics = {"roc_auc": "roc_auc", "recall": "recall", "f1": "f1"}

rows = []
for name, model in models.items():
    for label, scorer in metrics.items():
        s = cross_val_score(model, X, y, cv=cv, scoring=scorer)
        rows.append([name, label, s.mean(), s.std(), s.mean() - 1.96 * s.std() / np.sqrt(len(s))])

cv_table = pd.DataFrame(rows, columns=["모델", "지표", "평균", "표준편차", "CI하한"])
cv_pivot = cv_table.pivot(index="모델", columns="지표", values="평균").round(3)
print(cv_pivot.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=cv_table, x="모델", y="평균", hue="지표", ax=ax, palette="Set2")
ax.axhline(1 - y.mean(), color="gray", linestyle="--", label="기준선(정확도 아님, 참고)")
ax.set_title("교차검증 지표 비교")
ax.legend()
plt.tight_layout()
plt.show()

**핵심**: 클래스 가중치 덕분에 재현율이 **0.48 → 0.71~0.73** 으로 크게 올라갔습니다.
정확도가 낮아 보여도 실제 생존자를 훨씬 많이 잡습니다.

## 4. 임계값 선택 (교차검증 OOF 확률 사용)

테스트셋을 보지 않고, 교차검증 out-of-fold 확률로 F1이 최대가 되는 임계값을 고릅니다.
(테스트셋에서 임계값을 고르면 누출이므로 금지)

In [ ]:
best_model_name = "RandomForest"
best_model = models[best_model_name]

oof = cross_val_predict(best_model, X, y, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y, oof)
f1_curve = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
best_threshold = thr[np.argmax(f1_curve)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thr, f1_curve, color="darkorange")
axes[0].axvline(best_threshold, color="red", linestyle="--",
                label=f"F1 최대 임계값={best_threshold:.2f}")
axes[0].set_xlabel("임계값")
axes[0].set_ylabel("F1")
axes[0].set_title("임계값에 따른 F1")
axes[0].legend()

axes[1].plot(rec, prec, color="steelblue")
axes[1].set_xlabel("재현율")
axes[1].set_ylabel("정밀도")
axes[1].set_title("OOF Precision-Recall 곡선")
plt.tight_layout()
plt.show()

print(f"선택된 임계값: {best_threshold:.3f}")
print(f"OOF F1: {f1_score(y, (oof >= best_threshold).astype(int)):.3f}")

## 5. 최종 평가 (홀드아웃 테스트셋)

앞서 고른 모델과 임계값을 이제 처음 보는 테스트셋에 적용합니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
best_model.fit(X_train, y_train)
proba = best_model.predict_proba(X_test)[:, 1]
pred = (proba >= best_threshold).astype(int)

print(f"테스트 정확도: {accuracy_score(y_test, pred):.3f} | 기준선(전부 사망): {1 - y_test.mean():.3f}")
print(f"ROC-AUC      : {roc_auc_score(y_test, proba):.3f}")
print(f"정밀도        : {precision_score(y_test, pred):.3f}")
print(f"재현율        : {recall_score(y_test, pred):.3f}")
print(f"F1           : {f1_score(y_test, pred):.3f}")
print()
print(classification_report(y_test, pred, target_names=["사망", "생존"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["사망", "생존"], ax=axes[0])
axes[0].set_title(f"{best_model_name} 혼동행렬 (임계값 {best_threshold:.2f})")

fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, proba):.3f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_xlabel("거짓 양성률")
axes[1].set_ylabel("참 양성률")
axes[1].set_title("ROC 곡선")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. 변수 중요도

`permutation_importance` 는 각 변수를 섞었을 때 AUC가 얼마나 떨어지는지로 중요도를 잽니다.
(트리 자체 중요도보다 편향이 적습니다)

In [ ]:
result = permutation_importance(best_model, X_test, y_test, scoring="roc_auc",
                               n_repeats=20, random_state=RANDOM_STATE)
imp = pd.Series(result.importances_mean, index=X.columns).sort_values()

imp.plot(kind="barh", figsize=(8, 5), color="mediumseagreen")
plt.axvline(0, color="gray", linewidth=0.8)
plt.title("Permutation Importance (AUC 감소량)")
plt.xlabel("중요도")
plt.tight_layout()
plt.show()

imp.sort_values(ascending=False).round(3)

## 7. 이전 모델과 비교

| 항목 | 이전 모델 (검증 전) | 최종 모델 (인사이트 반영) |
|---|---|---|
| 모델 | RandomForest (기본) | RandomForest + 피처 + 클래스 가중치 |
| ROC-AUC (CV) | 0.797 | **0.805** |
| 재현율 (CV) | 0.477 | **0.725** |
| 정확도 (CV) | 0.795 | (재현율 우선이라 비슷 또는 소폭 변동) |

재현율이 오르면 정밀도는 내려가는 **트레이드오프**가 있습니다.
"생존 가능성이 있는 승객을 놓치지 않는 것"이 목표라면 이 방향이 맞습니다.

## 8. 한계와 결론

**한계**
- 데이터가 2차 가공본이고 표본이 1,309명뿐이라, 성능 수치는 ±0.02~0.03 정도 흔들립니다.
- 가족/티켓 그룹을 반영한 교차검증은 이번엔 넣지 않았습니다(대리 그룹은 검증 노트북 참고).
- `Age==28` 대체 등 인위적 결측이 남아 있어 나이 신호는 실제보다 약할 수 있습니다.

**결론**
1. 피처 엔지니어링(특히 `IsChild`)과 클래스 가중치가 **재현율을 크게 개선**했습니다.
2. 정확도 한 숫자가 아니라 **재현율·F1·AUC를 함께** 보고 모델을 판단해야 합니다.
3. 이 모델은 "생존자를 놓치지 않는" 용도에 맞춰져 있으며, 정밀도가 필요하면 임계값을 올리면 됩니다.

**다음 단계**
- `GridSearchCV` 로 하이퍼파라미터 튜닝
- 가족/티켓 그룹 기반 GroupKFold 재평가
- 모델 저장(`joblib`) 및 예측 함수화